# Workshop: Tunix-Med · Part 2: Baseline Evaluation

Before we fine-tune, we must know where we start. This notebook evaluates the **Base Gemma 3 1B** model on the cardiology dataset we just built.

### Evaluation Metrics
1. **Keyword F1 (TF-IDF weighted)**: Measures how well the model uses specific medical terminology.
2. **Semantic Similarity**: Uses a sentence embedding model to check if the meaning matches the reference.
3. **AI Judge (Qwen 7B)**: A larger LLM acts as an expert cardiologist to score the answer's clinical accuracy (1-10).
4. **Corpus Perplexity**: Measures how "surprised" the model is by the ground truth answers (token-weighted).


In [1]:
import os
import warnings
import logging
import math

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)
print(f"Device: {device}  |  dtype: {dtype}")

Device: cuda  |  dtype: torch.bfloat16


## 1 · Load Base Model & Dataset

### Tech Note: The Chat Template
Modern LLMs are trained with specific special tokens to separate roles (e.g., `<|im_start|>user`). We use `tokenizer.apply_chat_template` to ensure that our evaluation prompts match the exact format used during the model's original instruction tuning. This is crucial for obtaining fair and accurate baseline results.

In [2]:
MODEL_NAME = "google/gemma-3-270m-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map="auto",
)
model.eval()

DATASET_ID = "lmassaron/medical-cardiology-qa"
full_ds = load_dataset(DATASET_ID, split="train")
n = len(full_ds)

# Sample 100 questions from the 10% eval split
EVAL_SPLIT, SEED, N_EVAL_QS = 0.1, 42, 100
rng = np.random.default_rng(SEED)
all_idx = rng.permutation(n)
eval_idx = all_idx[int(n * (1.0 - EVAL_SPLIT)) :]


def extract_qa(example):
    msgs = example["messages"]
    return {
        "question": next(m["content"] for m in msgs if m["role"] == "user"),
        "answer": next(m["content"] for m in msgs if m["role"] == "assistant"),
    }


rng2, qa_pairs, seen_prefixes = np.random.default_rng(SEED + 1), [], set()
for idx in rng2.permutation(eval_idx):
    if len(qa_pairs) >= N_EVAL_QS:
        break
    ex = extract_qa(full_ds[int(idx)])
    if len(ex["answer"]) < 25:
        continue
    prefix = " ".join(ex["question"].lower().split()[:4])
    if prefix in seen_prefixes:
        continue
    seen_prefixes.add(prefix)
    qa_pairs.append({"question": ex["question"], "answer": ex["answer"]})

data = pd.DataFrame(qa_pairs)
print(f"Sampled {len(data)} questions.")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Sampled 100 questions.


## 2 · Baseline Inference & Fixed Perplexity

### Tech Note: Understanding Perplexity (PPL)
Perplexity measures how "surprised" a model is by a sequence of text. 
- Mathematically, it is the exponential of the cross-entropy loss: $PPL = e^{Loss}$.
- **Lower is better**: A lower PPL means the model predicted the next tokens in the ground-truth answer with high confidence.
- **Corpus-level PPL**: We weight the loss by the number of tokens in each answer to prevent short, easy-to-predict answers from biasing the results.

In [3]:
SYSTEM_PROMPT = "You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology."
results_list = []

total_loss_bits = 0.0
total_tokens = 0

for _, row in tqdm(data.iterrows(), total=len(data), desc="Baseline Inference"):
    # 1. Generation
    encoded = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]},
        ],
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    prompt_len = encoded["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    generated = tokenizer.decode(outputs[0, prompt_len:], skip_special_tokens=True).strip()

    # 2. Perplexity (True Corpus PPL)
    full_ids_ppl = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        return_tensors="pt",
        return_dict=True,
    )["input_ids"].to(device)
    labels = full_ids_ppl.clone()
    labels[:, :prompt_len] = -100
    with torch.no_grad():
        out = model(full_ids_ppl, labels=labels)
        loss_i = out.loss.item()

    n_answer_tokens = (labels != -100).sum().item()
    total_loss_bits += loss_i * n_answer_tokens
    total_tokens += n_answer_tokens

    perplexity_i = math.exp(min(loss_i, 20.0))
    results_list.append(
        {
            "question": row["question"],
            "expected_answer": row["answer"],
            "generated_answer": generated,
            "perplexity": perplexity_i,
        }
    )

results_df = pd.DataFrame(results_list)
corpus_ppl = math.exp(total_loss_bits / total_tokens)
print(f"Inference complete. Corpus Perplexity: {corpus_ppl:.2f}")

del model
torch.cuda.empty_cache()

Baseline Inference:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Inference complete. Corpus Perplexity: 21.68


## 3 · Scoring

### Tech Note: The LLM-as-a-Judge Pattern
Traditional metrics like ROUGE or BLEU often fail to capture clinical nuances. We use **Qwen 2.5 7B** as a "Judge" to evaluate the model's output against a reference. By giving the Judge a clear medical rubric and a 1-10 scale, we obtain a score that better reflects factual correctness and medical reasoning.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

_tfidf = TfidfVectorizer(analyzer="word", token_pattern=r"\b\w{4,}\b", sublinear_tf=True)
_tfidf.fit(results_df["expected_answer"].tolist())
_vocab, _idf = _tfidf.vocabulary_, _tfidf.idf_


def keyword_f1_tfidf(gen, exp):
    ref_kws = set(re.findall(r"\b\w{4,}\b", exp.lower()))
    gen_kws = set(re.findall(r"\b\w{4,}\b", gen.lower()))
    if not ref_kws:
        return 1.0

    def weighted_count(kws, universe):
        return sum(
            _idf[_vocab[w]] if w in _vocab else 1.0 for w in universe if w in kws
        )

    ref_weight = sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in ref_kws)
    gen_weight = (
        sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in gen_kws) if gen_kws else 0.0
    )
    if ref_weight == 0 or gen_weight == 0:
        return 0.0
    recall = weighted_count(gen_kws, ref_kws) / ref_weight
    precision = weighted_count(ref_kws, gen_kws) / gen_weight
    return float(
        (2 * precision * recall / (precision + recall))
        if (precision + recall) > 0
        else 0.0
    )


from sentence_transformers import SentenceTransformer, util

sim_model = SentenceTransformer("all-MiniLM-L6-v2")


def _raw_semantic(gen, exp):
    e1 = sim_model.encode(gen, convert_to_tensor=True, show_progress_bar=False)
    e2 = sim_model.encode(exp, convert_to_tensor=True, show_progress_bar=False)
    return float(util.pytorch_cos_sim(e1, e2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
judge_mdl = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, quantization_config=bnb_config, device_map="auto"
)


def ai_judge(question, generated, expected):
    prompt = (
        "You are an expert clinical cardiologist and medical education evaluator.\n"
        "Evaluate the 'Generated Answer' against the 'Reference Answer' for the given 'Question'.\n\n"
        "### SCORING RUBRIC (1-10):\n"
        "- 1: NO ANSWER (e.g., greetings, refusals, 'I am ready to help') or COMPLETELY WRONG.\n"
        "- 2-4: MAJOR ERRORS (significant inaccuracies or misses the key point).\n"
        "- 5-6: PARTIALLY CORRECT (covers some aspects but lacks depth or minor inaccuracies).\n"
        "- 7-8: MOSTLY CORRECT (aligns with the reference, but slightly incomplete or verbose).\n"
        "- 9-10: EXCELLENT (clinically precise, factual, accurate, and matches or improves on the reference).\n\n"
        "### INSTRUCTIONS:\n"
        "1. Penalize non-answers or conversational fluff with a score of 1.\n"
        "2. Score ONLY the medical accuracy and clinical relevance.\n"
        "3. Ignore chatbot niceties (e.g., 'I hope this helps').\n"
        "4. First write reasoning, then on the last line write ONLY: 'Score: [number]'\n\n"
        f"Question: {question}\n"
        f"Reference Answer: {expected}\n"
        f"Generated Answer: {generated}\n"
    )
    inp = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(judge_mdl.device)
    with torch.no_grad():
        out = judge_mdl.generate(**inp, max_new_tokens=150, do_sample=False)
    txt = judge_tok.decode(out[0, inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    
    # Improved extraction: look for "Score: X" on the last few lines
    lines = txt.splitlines()
    for line in reversed(lines):
        m = re.search(r"Score:\s*(\d+)", line)
        if m:
            return max(min(int(m.group(1)) / 10.0, 1.0), 0.1)
            
    # Fallback
    m = re.search(r"\b(\d+)\b", txt.splitlines()[-1])
    return max(min((int(m.group(1)) / 10.0) if m else 0.5, 1.0), 0.1)

results_df["keyword_score"] = results_df.apply(
    lambda r: keyword_f1_tfidf(r["generated_answer"], r["expected_answer"]), axis=1
)
results_df["_raw_sim"] = results_df.apply(
    lambda r: _raw_semantic(r["generated_answer"], r["expected_answer"]), axis=1
)
sim_min, sim_max = results_df["_raw_sim"].min(), results_df["_raw_sim"].max()
results_df["semantic_score"] = (
    (results_df["_raw_sim"] - sim_min) / (sim_max - sim_min)
).clip(0, 1)

print("Running AI Judge...")
scores = []
for _, r in tqdm(results_df.iterrows(), total=len(results_df), desc="AI Judge"):
    scores.append(ai_judge(r["question"], r["generated_answer"], r["expected_answer"]))
results_df["ai_judge_score"] = scores
results_df["final_score"] = results_df["keyword_score"] * 0.1 + results_df["semantic_score"] * 0.3 + results_df["ai_judge_score"] * 0.6

print("\n--- BASELINE RESULTS ---")
print(f"  Mean Keyword Score  : {results_df['keyword_score'].mean():.3f}")
print(f"  Mean Semantic Score : {results_df['semantic_score'].mean():.3f}")
print(f"  Mean AI Judge Score : {results_df['ai_judge_score'].mean():.3f}")
print(f"  Mean Final Score    : {results_df['final_score'].mean():.3f}")
print(f"  Corpus Perplexity   : {corpus_ppl:.2f}")
results_df.to_csv("medical_baseline_results.csv", index=False)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Running AI Judge...


AI Judge:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- BASELINE RESULTS ---
  Mean Keyword Score  : 0.208
  Mean Semantic Score : 0.620
  Mean AI Judge Score : 0.169
  Mean Final Score    : 0.308
  Corpus Perplexity   : 21.68


## 4 · Qualitative Analysis: Best & Worst Performers

Below we inspect the examples where the AI Judge gave the highest and lowest scores to understand the model's strengths and failure modes in its baseline state.

In [6]:
def display_examples(df, title, n=3, ascending=False):
    print(f"\n{'='*80}\n{title}\n{'='*80}")
    subset = df.sort_values("ai_judge_score", ascending=ascending).head(n)
    for i, (_, row) in enumerate(subset.iterrows()):
        print(f"\n[{i+1}] Score: {row['ai_judge_score']:.1f} | Question: {row['question']}")
        print(f"\nGenerated Answer:\n{row['generated_answer']}")
        print(f"\nReference Answer:\n{row['expected_answer']}")
        print(f"\n{'-'*80}")

display_examples(results_df, "TOP 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=False)
display_examples(results_df, "BOTTOM 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=True)


TOP 3 PERFORMING EXAMPLES (Baseline)

[1] Score: 0.7 | Question: Can mitral valve prolapse cause symptoms beyond palpitations and chest pain?

Generated Answer:
Yes, mitral valve prolapse can cause symptoms beyond palpitations and chest pain.

Reference Answer:
Yes, mitral valve prolapse can cause various symptoms including fatigue, dizziness, and syncope. These symptoms can vary in severity and frequency among individuals.

--------------------------------------------------------------------------------

[2] Score: 0.7 | Question: Is there any notable difference in the etiology of constrictive pericarditis between Western and developing countries?

Generated Answer:
Yes, there is a notable difference in the etiology of constrictive pericarditis between Western and developing countries.

Reference Answer:
Yes, in developing countries, tuberculosis is frequently identified as the leading cause of constrictive pericarditis, which is less common in Western countries.

-------------------

## 5 · Qualitative Analysis: Best & Worst Performers

Below we inspect the examples where the AI Judge gave the highest and lowest scores to understand the model's strengths and failure modes in its baseline state.

In [7]:
def display_examples(df, title, n=3, ascending=False):
    header = "=" * 80
    divider = "-" * 80
    print(f"\n{header}\n{title}\n{header}")
    subset = df.sort_values("ai_judge_score", ascending=ascending).head(n)
    for i, (_, row) in enumerate(subset.iterrows()):
        print(f"\n[{i+1}] Score: {row['ai_judge_score']:.1f} | Question: {row['question']}")
        print(f"\nGenerated Answer:\n{row['generated_answer']}")
        print(f"\nReference Answer:\n{row['expected_answer']}")
        print(f"\n{divider}")

display_examples(results_df, "TOP 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=False)
display_examples(results_df, "BOTTOM 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=True)



TOP 3 PERFORMING EXAMPLES (Baseline)

[1] Score: 0.7 | Question: Can mitral valve prolapse cause symptoms beyond palpitations and chest pain?

Generated Answer:
Yes, mitral valve prolapse can cause symptoms beyond palpitations and chest pain.

Reference Answer:
Yes, mitral valve prolapse can cause various symptoms including fatigue, dizziness, and syncope. These symptoms can vary in severity and frequency among individuals.

--------------------------------------------------------------------------------

[2] Score: 0.7 | Question: Is there any notable difference in the etiology of constrictive pericarditis between Western and developing countries?

Generated Answer:
Yes, there is a notable difference in the etiology of constrictive pericarditis between Western and developing countries.

Reference Answer:
Yes, in developing countries, tuberculosis is frequently identified as the leading cause of constrictive pericarditis, which is less common in Western countries.

-------------------